In [ ]:
import re
import pandas as pd
import time
from urllib.parse import urljoin, urlparse, parse_qs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

In [ ]:
KEYWORDS = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "ML",
    "deep learning",
    "large language model",
    "large language models",
    "LLM",
    "LLMs"
]

ai_pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(keyword) for keyword in KEYWORDS) + r")\b",
    re.IGNORECASE
)

In [ ]:
def scrape_isms_ai_presentations(driver, max_sessions=None, wait_timeout=20):

    main_url = (
        "https://submissions.mirasmart.com/ISMS2026/Itinerary/EventsAAG.aspx"
    )

    wait = WebDriverWait(driver, wait_timeout)

    # Extract abstract text from a presentation detail page
    def extract_abstract(body_text, speaker):

        lines = [
            line.strip()
            for line in body_text.splitlines()
            if line.strip()
        ]

        # The abstract ends before "Print Abstract"
        try:
            end_idx = next(
                i for i, line in enumerate(lines)
                if "Print Abstract" in line
            )
        except StopIteration:
            return ""

        # Find the author line containing the first speaker
        speaker_idx = next(
            (
                i for i, line in enumerate(lines[:end_idx])
                if speaker and speaker in line
            ),
            None
        )

        if speaker_idx is not None:
            abstract_lines = lines[speaker_idx + 1:end_idx]
            return " ".join(abstract_lines).strip()

        # Fallback: use the line after "Program Number"
        program_idx = next(
            (
                i for i, line in enumerate(lines[:end_idx])
                if "Program Number:" in line
            ),
            None
        )

        if program_idx is not None:
            abstract_lines = lines[program_idx + 2:end_idx]
            return " ".join(abstract_lines).strip()

        return ""

    # --------------------------------------------------
    # 1. Collect all session URLs from the main page
    # --------------------------------------------------

    driver.get(main_url)

    wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//a[contains(@href, 'EventDetail.aspx')]")
        )
    )

    session_elements = driver.find_elements(
        By.XPATH, "//a[contains(@href, 'EventDetail.aspx')]"
    )

    session_urls = list(dict.fromkeys(
        element.get_attribute("href")
        for element in session_elements
        if element.get_attribute("href")
    ))

    if max_sessions is not None:
        session_urls = session_urls[:max_sessions]

    print(f"Found {len(session_urls)} sessions.")

    # --------------------------------------------------
    # 2. Visit each session and collect presentations
    # --------------------------------------------------

    all_presentations = []

    for session_num, session_url in enumerate(session_urls, start=1):

        event_id = parse_qs(
            urlparse(session_url).query
        ).get("evt", [None])[0]

        if not event_id:
            continue

        try:
            # Return to the main page before opening the next session
            driver.get(main_url)

            session_link = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        f"//a[contains(@href, 'evt={event_id}')]"
                    )
                )
            )

            # Clicking the session link is necessary
            session_link.click()

            wait.until(
                EC.presence_of_element_located(
                    (By.CLASS_NAME, "ContentBox")
                )
            )

            presentation_boxes = driver.find_elements(
                By.CLASS_NAME, "ContentBox"
            )

            # Collect presentation information before navigating away
            session_presentations = []

            for box in presentation_boxes:

                title = box.find_element(
                    By.CLASS_NAME, "Title"
                ).text.strip()

                author_block = box.find_element(
                    By.CLASS_NAME, "AuthorBlock"
                )

                bold_names = author_block.find_elements(
                    By.TAG_NAME, "b"
                )

                speaker = (
                    bold_names[0].text.strip()
                    if bold_names else None
                )

                # The anchor wraps the ContentBox
                presentation_url = box.find_element(
                    By.XPATH, "./ancestor::a[1]"
                ).get_attribute("href")

                presentation_url = urljoin(
                    driver.current_url,
                    presentation_url
                )

                session_presentations.append({
                    "title": title,
                    "speaker": speaker,
                    "link": presentation_url
                })

            print(
                f"Session {session_num}/{len(session_urls)}: "
                f"{len(session_presentations)} presentations found."
            )

            # --------------------------------------------------
            # 3. Visit each presentation detail page
            # --------------------------------------------------

            for presentation in session_presentations:

                try:
                    driver.get(presentation["link"])

                    wait.until(
                        lambda d: "Print Abstract" in
                        d.find_element(By.TAG_NAME, "body").text
                    )

                    body_text = driver.find_element(
                        By.TAG_NAME, "body"
                    ).text

                    abstract = extract_abstract(
                        body_text,
                        presentation["speaker"]
                    )

                    # --------------------------------------------------
                    # 4. Keep only AI-related abstracts
                    # --------------------------------------------------

                    if ai_pattern.search(abstract):

                        all_presentations.append({
                            "title": presentation["title"],
                            "speaker": presentation["speaker"],
                            "abstract": abstract,
                            "link": presentation["link"]
                        })

                except Exception as e:
                    print(
                        f"Error processing presentation "
                        f"'{presentation['title']}': {e}"
                    )

        except Exception as e:
            print(f"Error processing session {event_id}: {e}")

    # --------------------------------------------------
    # 5. Create the final DataFrame
    # --------------------------------------------------

    df = pd.DataFrame(
        all_presentations,
        columns=["title", "speaker", "abstract", "link"]
    )

    print("\nScraping completed.")
    print(f"AI-related presentations found: {len(df)}")

    return df

In [ ]:
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

In [ ]:
df_ai = scrape_isms_ai_presentations(driver)
df_ai

In [ ]:
df_ai.to_csv('isms_conf_2026.csv')